# 📊 Notebook 1 — Exploratory Data Analysis

Understand the Superstore dataset before building any models.

## 1.1 Load Libraries & Data

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.join('..', 'src'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from data_preprocessing import load_and_clean, aggregate_monthly

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline


In [ ]:
df = load_and_clean('../data/train.csv')
print("Shape:", df.shape)
df.head()


## 1.2 Basic Statistics

In [ ]:
print(df.dtypes)
print()
print(df[['Sales']].describe())


## 1.3 Monthly Aggregation

In [ ]:
monthly = aggregate_monthly(df)
print(f"Monthly records: {len(monthly)}")
monthly.head(10)


## 1.4 Sales Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['ds'], monthly['y'], color='#4C72B0', linewidth=2, marker='o', markersize=4)
ax.fill_between(monthly['ds'], monthly['y'], alpha=0.15, color='#4C72B0')
ax.set_title('Monthly Total Sales — Superstore (2015–2018)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Sales ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/monthly_sales_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("Peak month:", monthly.loc[monthly['y'].idxmax(), 'ds'].strftime('%b %Y'))
print(f"Peak sales: ${monthly['y'].max():,.0f}")


## 1.5 Sales by Category

In [ ]:
cat_monthly = aggregate_monthly(df, group_cols=['Category'])
pivot = cat_monthly.pivot_table(index='ds', columns='Category', values='y', aggfunc='sum')

fig, ax = plt.subplots(figsize=(13, 4))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], linewidth=2, label=col, marker='o', markersize=3)
ax.set_title('Monthly Sales by Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Sales ($)'); ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()


## 1.6 Sales by Region

In [ ]:
region_total = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
region_total.plot(kind='bar', ax=ax, color=['#4C72B0','#DD8452','#55A868','#C44E52'], edgecolor='white')
ax.set_title('Total Sales by Region', fontsize=13, fontweight='bold')
ax.set_ylabel('Sales ($)'); ax.set_xlabel('')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## 1.7 Seasonality Check — Average Sales by Month

In [ ]:
monthly['month_name'] = monthly['ds'].dt.strftime('%b')
monthly['month_num']  = monthly['ds'].dt.month
avg_by_month = monthly.groupby(['month_num', 'month_name'])['y'].mean().reset_index().sort_values('month_num')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(avg_by_month['month_name'], avg_by_month['y'], color='#4C72B0', edgecolor='white')
ax.set_title('Average Monthly Sales (Seasonality Pattern)', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Sales ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
print("Highest sales month:", avg_by_month.loc[avg_by_month['y'].idxmax(), 'month_name'])


## 1.8 Rolling Mean & Std (Stationarity Check)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly['ds'], monthly['y'], label='Actual', color='black', linewidth=1.5)
ax.plot(monthly['ds'], monthly['y'].rolling(3).mean(), label='3M Rolling Mean', color='blue', linewidth=2)
ax.plot(monthly['ds'], monthly['y'].rolling(3).std(), label='3M Rolling Std', color='orange', linewidth=2, linestyle='--')
ax.set_title('Rolling Statistics — Stationarity Check', fontsize=13, fontweight='bold')
ax.set_ylabel('Sales ($)'); ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()


## 1.9 Key Takeaways

- Sales show a clear **upward trend** from 2015–2018
- **Q4 (Nov–Dec)** is consistently the strongest quarter — holiday seasonality
- **Technology** is the highest-revenue category
- **West region** leads in total sales
- The series is **non-stationary** (rising mean) → models must handle trend

➡️ Proceed to **Notebook 2 — Prophet Model**